In [0]:
dbutils.widgets.dropdown(name='environment',defaultValue='dev',choices=['dev','qa','prod'],label='select Environment')

env=dbutils.widgets.get('environment')

bronzeTablName = f"saleslake_{env}.bronze_{env}.rawDiscount"
print(bronzeTablName)
silverTablName=f"saleslake_{env}.silver_{env}.cleaneddiscount"
print(silverTablName)




In [0]:
# spark.sql(f"""
# INSERT INTO {silverTablName}
# SELECT DISTINCT
#     CAST(TRIM(discount_id) AS INT)            AS discount_id,
#     UPPER(TRIM(discount_code))                AS discount_code,
#     TRIM(discount_name)                       AS discount_name,
#     UPPER(TRIM(discount_type))                AS discount_type,
#     CAST(TRIM(discount_value) AS DECIMAL(10,2)) AS discount_value,
#     CAST(TRIM(min_purchase_amount) AS DECIMAL(18,2)) AS min_purchase_amount,
#     CAST(TRIM(max_discount_amount) AS DECIMAL(18,2)) AS max_discount_amount,
#     TO_DATE(TRIM(valid_from), 'yyyy-MM-dd')   AS valid_from,
#     TO_DATE(TRIM(valid_to),   'yyyy-MM-dd')   AS valid_to,
#     UPPER(TRIM(applicable_segment))           AS applicable_segment,
#     UPPER(TRIM(applicable_category))          AS applicable_category,
#     CAST(TRIM(usage_limit_per_customer) AS INT) AS usage_limit_per_customer,
#     CAST(TRIM(total_usage_limit) AS INT)      AS total_usage_limit,
#     UPPER(TRIM(is_active))                    AS is_active,
#     TO_DATE(TRIM(created_date), 'yyyy-MM-dd') AS created_date,
#     CURRENT_TIMESTAMP() AS ingest_ts
# FROM {bronzeTablName}
# ORDER BY discount_id
# """)

# # df.write.format("delta").mode("append").save(silverTablName)

# # print(f"Silver load complete for {silverTablName}")
# # spark.read.format("delta").load(silverTablName).count()

In [0]:
from pyspark.sql import functions as F

# Read bronze table
bronze_df = spark.table(bronzeTablName)

# Transformations
bronze_df = (
    bronze_df
    .withColumn("discount_id", F.col("discount_id").cast("int"))
    .withColumn("discount_code", F.upper(F.trim(F.col("discount_code"))))
    .withColumn("discount_name", F.trim(F.col("discount_name")))
    .withColumn("discount_type", F.upper(F.trim(F.col("discount_type"))))
    .withColumn("discount_value", F.col("discount_value").cast("decimal(10,2)"))
    .withColumn("min_purchase_amount", F.col("min_purchase_amount").cast("decimal(18,2)"))
    .withColumn("max_discount_amount", F.col("max_discount_amount").cast("decimal(18,2)"))
    .withColumn("valid_from", F.to_date(F.trim(F.col("valid_from")), "yyyy-MM-dd"))
    .withColumn("valid_to", F.to_date(F.trim(F.col("valid_to")), "yyyy-MM-dd"))
    .withColumn("applicable_segment", F.upper(F.trim(F.col("applicable_segment"))))
    .withColumn("applicable_category", F.upper(F.trim(F.col("applicable_category"))))
    .withColumn("usage_limit_per_customer", F.col("usage_limit_per_customer").cast("int"))
    .withColumn("total_usage_limit", F.col("total_usage_limit").cast("int"))
    .withColumn("is_active", F.upper(F.trim(F.col("is_active"))))
    .withColumn("created_date", F.to_date(F.trim(F.col("created_date")), "yyyy-MM-dd"))
    .withColumn("ingest_ts", F.current_timestamp())
)


# Incremental load condition - handle first load when table doesn't exist
try:
    max_ingest_ts = spark.read.format("delta").load(silverTablName).agg(
        F.coalesce(F.max("ingest_ts"), F.to_timestamp(F.lit("1990-01-01"), "yyyy-MM-dd"))
    ).collect()[0][0]
    df_filtered = bronze_df.filter(F.col("ingest_ts") > max_ingest_ts).distinct()
except Exception as e:
    # Table doesn't exist yet - first load, use all data
    print(f"First load - table doesn't exist yet. Loading all data.")
    df_filtered = bronze_df.distinct()

# Write to silver table
(df_filtered.write
   .format("delta")
   .mode("append")                   # append new incremental rows
   .option("mergeSchema", "true")    # allow schema evolution
   .saveAsTable(silverTablName))

# print(f"Silver load complete for {silverTablName}")
# spark.sql(f"SELECT COUNT(*) AS row_count FROM {silverTablName}").display()


In [0]:
%sql
SELECT * FROM saleslake_dev.silver_dev.cleaneddiscount ;
SELECT * FROM saleslake_prod.silver_prod.cleaneddiscount 